# 07 – External Check on Real GLOF Events (HMAGLOFDB)

This notebook only **checks** the final model against real documented GLOF events in Pakistan. 

### 1. Load HMAGLOFDB and the final lake dataset

In [1]:
import pandas as pd
import numpy as np
import joblib
from pathlib import Path
from sklearn.neighbors import BallTree
from sklearn.base import clone
from sklearn.metrics import roc_auc_score

PROJECT_ROOT = Path("..")
hma = pd.read_csv(PROJECT_ROOT / "data" / "raw" / "HMAGLOFDB.csv", encoding="latin1", sep=None, engine="python")
lakes = pd.read_csv(PROJECT_ROOT / "data" / "final" / "glofguard_training_data.csv")

print("HMAGLOFDB shape:", hma.shape)
print("Lake dataset shape:", lakes.shape)

HMAGLOFDB shape: (768, 59)
Lake dataset shape: (8806, 13)


### 2. Filter to Pakistan GLOF events

In [2]:
pk_events = hma[hma["Country"].astype(str).str.strip().str.lower() == "pakistan"].copy()
print("Pakistan GLOF events:", len(pk_events))
pk_events[["Year_exact", "Month", "Day", "Lat_lake", "Lon_lake", "Lake_name", "Glacier_name"]].head(20)

Pakistan GLOF events: 151


,Year_exact,Month,Day,Lat_lake,Lon_lake,Lake_name,Glacier_name
2,1661.0,NaN,NaN,35.687000,75.907000,Unnamed,Biafo
4,1833.0,NaN,NaN,36.824000,74.369000,Unknown,Yashkuk-Yaz
9,1844.0,NaN,NaN,36.621400,74.079060,Karambar,Unnamed
11,1850.0,NaN,NaN,35.232000,74.722000,Unknown,Tarshing
13,1860.0,NaN,NaN,36.621400,74.079060,Karambar,Unnamed
14,1860.0,7.0,NaN,35.668000,75.378000,Unknown,Unnamed
16,1865.0,NaN,NaN,36.794600,73.994940,Sokhter Rabot,Unnamed
17,1865.0,NaN,NaN,36.621400,74.079060,Karambar,Unnamed
20,1873.0,NaN,NaN,36.510556,74.850591,Unnamed,Batura
25,1882.0,9.0,NaN,36.344239,75.474953,Unnamed,Khurdopin


### 3. Match each event to the nearest lake

In [3]:
event_coords = np.radians(pk_events[["Lat_lake", "Lon_lake"]].values)
lake_coords = np.radians(lakes[["latitude", "longitude"]].values)

tree = BallTree(lake_coords, metric="haversine")
distances, indices = tree.query(event_coords, k=1)

pk_events["matched_sample_id"] = lakes.iloc[indices.flatten()]["sample_id"].values
pk_events["match_distance_km"] = distances.flatten() * 6371

for km in [1, 2, 3]:
    m = pk_events[pk_events["match_distance_km"] <= km]
    print(km, "km ->", len(m), "events,", m["matched_sample_id"].nunique(), "lakes")

1 km -> 27 events, 14 lakes
2 km -> 80 events, 20 lakes
3 km -> 88 events, 24 lakes


### 4. Load the final model

In [4]:
import joblib

# Load the saved final model and its feature list
saved = joblib.load(PROJECT_ROOT / "artifacts" / "final_model.joblib")
final_model = saved["pipeline"]
feature_cols = saved["features"]
print("Model loaded with", len(feature_cols), "features")

Model loaded with 7 features


### 5. Fair external validation (unseen lakes)

The final model already saw the event lakes during training, so scoring them with it would be biased. Instead we retrain the same pipeline **without** the 0.25° grid cells that contain event lakes (same grouping as notebook 06), then score those lakes as truly unseen data. We repeat this for 1, 2 and 3 km matching distances: a larger distance gives more test lakes but less certain matches.

In [5]:
def fair_test(km):
    m = pk_events[pk_events["match_distance_km"] <= km]
    event_ids = m["matched_sample_id"].unique()

    df = lakes.copy()
    df["near_real_event"] = df["sample_id"].isin(event_ids)

    # Same geographic grouping as notebook 06 (0.25 degree grid cells)
    df["geo_group"] = ((df["latitude"] // 0.25).astype(int).astype(str) + "_" +
                       (df["longitude"] // 0.25).astype(int).astype(str))
    held = df.loc[df["near_real_event"], "geo_group"].unique()

    train = df[~df["geo_group"].isin(held)]
    test = df[df["geo_group"].isin(held)]

    # Fresh copy of the same pipeline, trained WITHOUT the event lakes' cells
    model = clone(final_model)
    model.fit(train[feature_cols], train["label"])

    test_scores = model.predict_proba(test[feature_cols])[:, 1]
    all_scores = model.predict_proba(df[feature_cols])[:, 1]
    pct = pd.Series(all_scores, index=df.index).rank(pct=True)
    ev = df["near_real_event"]

    auc = (roc_auc_score(test["near_real_event"], test_scores)
           if test["near_real_event"].nunique() == 2 else np.nan)

    return {
        "match_km": km,
        "events": len(m),
        "event_lakes": len(event_ids),
        "were_label_1_in_training": int(df.loc[ev, "label"].sum()),
        "held_out_cells": len(held),
        "training_lakes": len(train),
        "avg_percentile_%": round(pct[ev].mean() * 100, 1),   # random = 50
        "share_in_top_20%": round((pct[ev] >= 0.8).mean() * 100, 1),  # random = 20
        "auc_held_out_cells": round(auc, 3),                  # random = 0.5
    }

fair_results = pd.DataFrame([fair_test(km) for km in [1, 2, 3]])
fair_results

,match_km,events,event_lakes,were_label_1_in_training,held_out_cells,training_lakes,avg_percentile_%,share_in_top_20%,auc_held_out_cells
0,1,27,14,14,10,8134,77.1,57.1,0.423
1,2,80,20,20,15,7829,61.5,25.0,0.500
2,3,88,24,24,16,7788,55.4,16.7,0.498


### How to read the fair test

- **avg_percentile_%**: where the real event lakes rank among all 8,806 lakes. Random = 50%. Higher is better (e.g. 75% means event lakes sit in the top quarter on average).
- **share_in_top_20%**: how many event lakes the model puts in its top 20%. Random = 20%.
- **auc_held_out_cells**: inside the held-out areas, can the model tell event lakes apart from other lakes? Random = 0.5.
- If these stay clearly above random at 1, 2 and 3 km, the model is picking up real GLOF risk, not just copying the 5 km labels.

**Limitation:** only a small number of real events could be matched to lakes (GLOFs are rare and many records have rough coordinates), so this is a supporting check. The main evaluation is the 3-fold geographic cross-validation in notebook 06.

### Conclusion

I used the HMAGLOFDB database, which lists 151 documented GLOF events in Pakistan. At a 1 km matching distance, I matched 27 events to 14 unique lakes in my GLOFGuard dataset (80 events / 20 lakes at 2 km, 88 events / 24 lakes at 3 km). All of these lakes were already labelled high-risk in my training data, so testing them with my final model would be biased.

To make the test fair, I retrained my model without the 0.25° grid cells that contain these lakes, and then scored the lakes as unseen data.

| Match | Lakes | Avg percentile (random 50%) | In top 20% (random 20%) | AUC in held-out cells (random 0.5) |
|---|---|---|---|---|
| 1 km | 14 | 77.1% | 57.1% | 0.42 |
| 2 km | 20 | 61.5% | 25.0% | 0.50 |
| 3 km | 24 | 55.4% | 16.7% | 0.50 |

**Interpretation:** With the most reliable 1 km matches, the real GLOF lakes ranked well above random across the whole region (77th percentile, 57% in the top 20%). This shows that my model captures regional risk patterns. However, inside their own grid cells, my model could not separate the event lakes from neighbouring lakes (AUC 0.42), and the signal became weaker at 2 km and 3 km, where the matches are less certain.

**Limitations:** I only had 14–24 event lakes to test, so these results are uncertain. My training labels are based on closeness to known lakes, not on confirmed outbursts.

**Conclusion:** I found that GLOFGuard is useful for identifying high-risk regions for monitoring, but it should not be used to predict which specific lake will burst. I therefore describe GLOFGuard as a GLOF susceptibility / risk-ranking model, not a real-time prediction system.